In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("Pokemon_Bronze_Silver_Gold")
    .master("local[*]")
    .getOrCreate()
)

In [4]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv("data/pokeData.csv")
)

In [5]:
raw_df.show(5, truncate=False)

+----------+----------+--------+-------------------+----------+
|pokemon_id|name      |height_m|updated_at         |batch_date|
+----------+----------+--------+-------------------+----------+
|1         |Bulbasaur |0.7     |2026-10-10 09:00:00|2026-10-10|
|2         |Ivysaur   |1.0     |2026-10-10 09:15:00|2026-10-10|
|3         |Venusaur  |2.0     |2026-10-10 09:30:00|2026-10-10|
|4         |Charmander|0.6     |2026-10-10 10:00:00|2026-10-10|
|5         |Charmeleon|1.1     |2026-10-10 10:15:00|2026-10-10|
+----------+----------+--------+-------------------+----------+
only showing top 5 rows


In [6]:
raw_df.write.mode("overwrite").parquet("output/bronze")

In [8]:
bronze_df = spark.read.parquet("output/bronze")

In [9]:
from pyspark.sql import functions as F

In [10]:
clean_df = (
    bronze_df
    .withColumn("pokemon_id", F.col("pokemon_id").cast("int"))
    .withColumn("height_m", F.col("height_m").cast("double"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .withColumn("batch_date", F.to_date("batch_date"))
    .withColumn("name", F.trim(F.col("name")))
)


In [11]:
clean_df = clean_df.filter(
    F.col("pokemon_id").isNotNull()
    & (F.col("pokemon_id") > 0)
    & F.col("name").isNotNull()
    & (F.col("name") != "")
    & F.col("height_m").isNotNull()
    & (F.col("height_m") >= 0)
    & F.col("updated_at").isNotNull()
)

In [13]:
from pyspark.sql import Window

In [14]:
window = Window.partitionBy("pokemon_id").orderBy(
    F.col("updated_at").desc(),
    F.col("batch_date").desc()
)

In [15]:
silver_df = (
    clean_df
    .withColumn("row_num", F.row_number().over(window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [16]:
silver_df.write.mode("overwrite").parquet("output/silver")

In [17]:
silver_df.orderBy("pokemon_id").show(truncate=False)

+----------+----------+--------+-------------------+----------+
|pokemon_id|name      |height_m|updated_at         |batch_date|
+----------+----------+--------+-------------------+----------+
|1         |Bulbasaur |0.7     |2026-10-10 09:00:00|2026-10-10|
|2         |Ivysaur   |1.0     |2026-10-10 09:15:00|2026-10-10|
|3         |Venusaur  |2.0     |2026-10-12 08:00:00|2026-10-12|
|4         |Charmander|0.8     |2026-10-14 08:00:00|2026-10-14|
|5         |Charmeleon|1.2     |2026-10-12 09:00:00|2026-10-12|
|6         |Charizard |1.7     |2026-10-10 10:30:00|2026-10-10|
|7         |Squirtle  |0.5     |2026-10-11 09:00:00|2026-10-11|
|8         |Wartortle |1.1     |2026-10-14 08:30:00|2026-10-14|
|9         |Blastoise |1.6     |2026-10-11 09:30:00|2026-10-11|
|10        |Caterpie  |0.3     |2026-10-11 10:00:00|2026-10-11|
|11        |Metapod   |0.7     |2026-10-12 09:30:00|2026-10-12|
|12        |Butterfree|1.1     |2026-10-12 10:00:00|2026-10-12|
|13        |Weedle    |0.3     |2026-10-

In [18]:
silver_df = spark.read.parquet("output/silver")

In [21]:
silver_df.filter(
    F.col("pokemon_id").isNull()
).count()

0

In [22]:
silver_df.filter(
    F.col("name").isNull() | (F.trim(F.col("name")) == "")
).count()

0

In [23]:
silver_df.filter(
    F.col("height_m").isNull() | (F.col("height_m") < 0)
).count()

0

In [24]:
silver_df.groupBy("pokemon_id")\
    .count()\
    .filter(F.col("count") > 1)\
    .count()

0

In [25]:
gold_df = (
    silver_df
    .groupBy("batch_date")
    .agg(
        F.count("*").alias("pokemon_count"),
        F.round(F.avg("height_m"), 2).alias("average_height_m"),
        F.max("height_m").alias("maximum_height_m")
    )
    .orderBy("batch_date")
)


In [26]:
gold_df.write.mode("overwrite").parquet("output/gold")

In [27]:
gold_df.show(truncate=False)

+----------+-------------+----------------+----------------+
|batch_date|pokemon_count|average_height_m|maximum_height_m|
+----------+-------------+----------------+----------------+
|2026-10-10|3            |1.13            |1.7             |
|2026-10-11|3            |0.8             |1.6             |
|2026-10-12|4            |1.25            |2.0             |
|2026-10-13|5            |0.66            |1.1             |
|2026-10-14|5            |0.88            |1.5             |
+----------+-------------+----------------+----------------+

